# Single factorization

Inspect the one-body and full single-factorization block encodings for
the 50-factor, 10-spin-orbital examples. The signatures and call graphs
below describe the locked model.

Adapted from Qualtran v0.7.0. Copyright 2023 Google LLC.
Licensed under Apache-2.0.

In [ ]:
from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.single_factorization_encoding_model import (
    source_single_factorization_block_encoding_cost,
    source_single_factorization_block_signature,
    source_single_factorization_one_body_cost,
    source_single_factorization_one_body_signature,
)
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg

def show_signature(signature, model):
    display(SVG(field_signature_svg([(model.name, signature, {})])))

def show_calls(model):
    display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
    print(estimate_component_resources(model))

## One-body block encoding

The three success and factor flags precede the bounded factor and spatial
orbital addresses. The signal-state declaration uses those selection ports,
and the target comprises two spin banks.

Inner PREPARE is followed by spin and pair symmetrization, SELECT, and
matching cleanup. The two lookup axes can be blocked independently in
PREPARE and PREPARE adjoint.

### Example

In [ ]:
auxiliaries, spin_orbitals, alias_bits, rotation_bits = 50, 10, 12, 7
body_ports = source_single_factorization_one_body_signature(auxiliaries, spin_orbitals)
body_model = source_single_factorization_one_body_cost(
    auxiliaries, spin_orbitals, alias_bits, rotation_bits
)

### Register signature

In [ ]:
show_signature(body_ports, body_model)

### Call graph

In [ ]:
show_calls(body_model)

## Full block encoding

Outer PREPARE and its adjoint surround two one-body calls with a
reflection between them. The source groups the three controls and the two
spatial indices. Its intermediate reflection is controlled by the outer
success bit. The source leaves alpha and epsilon unimplemented.

### Example

In [ ]:
block_ports = source_single_factorization_block_signature(auxiliaries, spin_orbitals)
block_model = source_single_factorization_block_encoding_cost(
    auxiliaries, spin_orbitals, alias_bits, rotation_bits
)

### Register signature

In [ ]:
show_signature(block_ports, block_model)

### Call graph

In [ ]:
show_calls(block_model)

### Executable coefficient example

The native coefficient-based binding implements the paper's signed
one-body and factor terms. It provides additional alias and sign words
and uses orbital-major system ordering. The native reflection is
conditioned on both outer success and nonzero factor. Here a small
one-spatial-orbital binding builds a separate instruction stream.

In [ ]:
from collections import Counter

from flagquantum.algorithms.primitives.single_factorization_block_encoding import (
    SingleFactorizationBlockEncoding,
)
from flagquantum.compiler.registers.layout.partition import partition_register
from flagquantum.dynamic import DynamicCircuit

op = SingleFactorizationBlockEncoding.from_coefficients(
    1, (-1.0,), (), outer_bits=1, inner_bits=1
)
logical_width = sum(width for _, width in op.layout)
work_count, _ = op.requirements()
total_width = logical_width + work_count
ports = partition_register(tuple(range(logical_width)), op.layout)
work = tuple(range(logical_width, total_width))
circuit = DynamicCircuit(total_width).append(op.make_on(ports, workspace=work))
emitted = Counter(gate.name for gate in circuit.to_ir().instructions)
print("Emitted T and T adjoints", emitted["t"] + emitted["tdg"])
print("Emitted measurements", emitted["measure"], "reusable work wires", work_count)
assert (emitted["t"] + emitted["tdg"], emitted["measure"], work_count) == (94, 13, 4)